# 04 — Feature Engineering
This notebook creates useful transaction and customer features from the raw columns.

## Learning objectives
- Create LineValue and calendar features
- Build customer summaries with `groupby().agg()`
- Understand conditional segmentation with `np.where()`

In [ ]:
import numpy as np
from retail_analysis.data_loader import load_retail_data
from retail_analysis.cleaning import clean_transactions, split_transaction_sets
from retail_analysis.features import add_transaction_features, create_customer_features

cleaned, _ = clean_transactions(load_retail_data())
sales, _, _ = split_transaction_sets(cleaned)
sales = add_transaction_features(sales)
sales[['Quantity', 'UnitPrice', 'LineValue', 'YearMonth', 'DayOfWeek', 'Hour']].head()

Multiplication between Pandas columns is vectorized, so a row-by-row loop is unnecessary. `YearMonth` includes the year, preventing the same month from different years from being combined.

In [ ]:
customers = create_customer_features(sales)
customers.head(10)

`groupby('CustomerID').agg(...)` combines rows for each customer. `nunique()` counts unique orders rather than invoice lines. The recency reference is one day after the latest eligible sales date. These features are not net of refunds.

In [ ]:
# np.where selects a label for each True/False condition without a loop
np.where(customers['OrderCount'].eq(1), 'One-time buyer', 'Repeat buyer')[:10]

In [ ]:
customers.groupby('CustomerSegment').agg(
    CustomerCount=('CustomerID', 'count'),
    AveragePurchaseValue=('TotalPurchaseValue', 'mean')
)

## Interpretation and practice questions
Anonymous transactions remain in sales totals but are absent from the customer table because they have no CustomerID. AverageOrderValue equals TotalPurchaseValue divided by unique OrderCount.

1. Compare the one-time and repeat customer counts.
2. Verify OrderCount for a customer whose invoice contains several line items.
3. Which customers have the lowest RecencyDays values?